In [1]:
import numpy as np
import numpyro
import numpyro.distributions as dist
from tqdm.notebook import tqdm

from modulars.plot_rr import plot_a_few_trajectories_1d, plot_mean_band_rrs_1d
from modulars import load_config, load_best_values
from modulars import exponential, print_model_info
from modulars import exp_lognormal_moments
transform_fn = exp_lognormal_moments


In [2]:
config_file = '../exp_config.json'
config = load_config(config_file)

lambda_like = config['lambda_like']
alpha_prior = config['alpha_prior']
beta_prior = config['beta_prior']
n_samples = config['n_samples']
max_iters = 100_000#config['max_iters']
data = exponential(n_samples, lambda_like)
print_model_info(
    "Gamma", "Exp", "Gamma",
    [alpha_prior, beta_prior], [lambda_like],
    data=data, n_samples=n_samples
)

Prior: Gamma([1.5, 1.0])
Likelihood: Exp([2.0])
Data: []


In [3]:
def model(y):
    lambd = numpyro.sample('lambd', dist.Gamma(alpha_prior, beta_prior))
    numpyro.sample('y', dist.Exponential(1/lambd), obs=y)


In [ ]:
from modulars import run_restart_1d
results = []
for seed in tqdm(range(100)):
    result = run_restart_1d(seed, model, data, 'lambd', 100_000, n_particles=100)
    results.append(result)

  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
from modulars import apply_traj_transform, save_rr_tracking_csv
TRACKING_CSV = "processed_tracking/rr_numpyro_gam_exp_tracking.csv"

single_means, single_stds, multi_means, multi_stds = \
apply_traj_transform(results, transform_fn, n_samples=10_000, seed=1, NOTEBOOK=True)
save_rr_tracking_csv(
    TRACKING_CSV,
    {"default": (single_means, single_stds, multi_means, multi_stds)},
)


In [ ]:
from modulars import load_rr_tracking_csv

TRACKING_CSV = "processed_tracking/rr_numpyro_gam_exp_tracking.csv"

single_means, single_stds, multi_means, multi_stds, x = load_rr_tracking_csv(
    TRACKING_CSV, scenario="default"
)
N, T = single_means.shape

# If we have "best" from config, these should be on theta-scale (0,1)
best_mu, best_std = load_best_values(
    config=config, transform=exp_lognormal_moments,
    n_samples=50_000, seed=1)

plot_a_few_trajectories_1d(
    [single_means, single_stds], [multi_means, multi_stds],
    best_mu, best_std, r'$\lambda$')
plot_mean_band_rrs_1d(
    single_means, single_stds, best_mu, best_std,
    x, r'$\lambda$', 1)
plot_mean_band_rrs_1d(
    multi_means, multi_stds, best_mu, best_std,
    x, r'$\lambda$', 100)
